In [29]:
import pandas as pd

df = pd.read_csv("data/titanic.csv")
df_filter = df[['pclass', 'age', 'sex']].copy()       # 不包含 survived
df_filter['age'].fillna(df_filter['age'].mean(), inplace=True)
y = df['survived']
x = pd.get_dummies(df_filter, columns=["pclass", 'sex'])

C:\Users\21830\AppData\Local\Temp\ipykernel_22488\1798247513.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_filter['age'].fillna(df_filter['age'].mean(), inplace=True)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
from sklearn import tree

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# 下面这行代码的含义：
# 创建一个决策树分类器（DecisionTreeClassifier）对象，命名为 clf。
# 该决策树使用"信息熵"作为划分标准（criterion='entropy'），
# 设置树的最大深度为3层（max_depth=3），
# 并且指定了随机种子为42（random_state=42），以保证结果可复现。
# 设置决策树的最大深度为3层，可以有效防止模型过拟合（overfitting）。
# 当决策树过深时，它可能会学习到训练集中的噪声，导致泛化能力变差。
# 限制树的深度，可以让树只学习到数据中最重要、最主要的特征划分，提升模型在新样本上的表现。
# clf = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
clf = DecisionTreeClassifier(
    random_state=42,         # 随机种子，保证结果可复现
    max_depth=6,             # 树的最大深度，防止过拟合
    min_samples_split=10,    # 内部节点再划分所需最小样本数
    min_samples_leaf=4,      # 叶子节点最少样本数，避免出现只含一个样本的叶子
    criterion='entropy'      # 划分标准，这里用信息增益（熵）
)

clf.fit(X_train, y_train)

# 下面两行代码的功能说明：
# 第一行：使用训练好的决策树分类器（clf）对测试集数据（X_test）进行预测，得到预测的标签（y_pred）。
y_pred = clf.predict(X_test)

# 第二行：通过准确率评估函数 accuracy_score，将模型预测值（y_pred）与测试集真实标签（y_test）进行比较，计算模型在测试集上的准确率（acc）。
acc = accuracy_score(y_test, y_pred)
print("acc:",acc)

acc: 0.8365019011406845


In [31]:
from sklearn.ensemble import RandomForestClassifier

# 参数解释：
# n_estimators：森林中树的数量，这里为100。
# random_state：随机种子，保证实验结果可复现，这里为42。
# max_depth：每颗决策树的最大深度，这里为6，限制树的增长防止过拟合。
# min_samples_split：一个节点再分裂所需的最小样本数，这里为10，增加该值可防止模型对小样本的过度拟合。
# min_samples_leaf：叶子节点最少样本数，这里为4，保证每个叶子节点至少包含4个样本。
# criterion：每次分裂的划分标准，这里为“entropy”，使用信息熵来度量。

rf_clf = RandomForestClassifier(
    n_estimators=100,         # 森林里树的数量
    random_state=42,          # 随机数种子，保证可复现
    max_depth=6,              # 每棵树的最大深度，防止过拟合
    min_samples_split=10,     # 内部节点再划分所需最小样本数
    min_samples_leaf=4,       # 叶子节点最少样本数
    criterion='entropy'       # 划分标准：信息熵
)

rf_clf.fit(X_train, y_train)
rf_y_pred = rf_clf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_y_pred)

print("随机森林准确率：", rf_acc)

随机森林准确率： 0.8250950570342205


In [33]:
param_grid = {
    'n_estimators': [100, 200, 300],        # 森林中树的数量
    'max_depth': [3, 5, 7, None],           # 树的最大深度，None表示不限制
    'criterion': ['gini', 'entropy'],       # 划分标准
    'max_features': ['sqrt', 'log2', None], # 每棵树最大特征数，常用sqrt(特征数)、log2(特征数)和全部特征
    'min_samples_split': [2, 5, 10],        # 内部节点再划分所需最小样本数
    'min_samples_leaf': [1, 2, 4],          # 叶子节点最少样本数
    'random_state': [42],                   # 保证可复现
    'bootstrap': [True, False]
}

from sklearn.model_selection import GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# 在训练集上训练
grid_search.fit(X_train, y_train)

# 输出最佳参数和最佳分数
print("最佳参数:", grid_search.best_params_)
print("最佳准确率: {:.4f}".format(grid_search.best_score_))

# 输出最佳max_features对应的特征数
best_max_features = grid_search.best_params_['max_features']
if best_max_features is None or best_max_features == 'auto':
    n_features_used = X_train.shape[1]
elif best_max_features == 'sqrt':
    import math
    n_features_used = int(math.sqrt(X_train.shape[1]))
elif best_max_features == 'log2':
    import math
    n_features_used = int(math.log2(X_train.shape[1]))
else:
    n_features_used = best_max_features

print(f"最优模型使用的特征数: {n_features_used}")

# 在测试集上评估最优模型
best_rf = grid_search.best_estimator_
test_acc = best_rf.score(X_test, y_test)
print("最优模型在测试集上的准确率: {:.4f}".format(test_acc))

c:\Users\21830\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


最佳参数: {'bootstrap': True, 'criterion': 'gini', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100, 'random_state': 42}
最佳准确率: 0.8276
最优模型使用的特征数: 2
最优模型在测试集上的准确率: 0.8213
